In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/diagnostics.py ──
"""DL Diagnostics Toolkit — clinical instruments for deep-network training.

Wraps PyTorch forward/backward hooks into a student-friendly API that
surfaces the four failure modes professionals must recognise:

    1. Stethoscope  — loss-curve shape (under/over-fit, instability)
    2. Blood Test   — gradient flow per layer (vanishing / exploding)
    3. X-Ray        — activation statistics & dead neurons
    4. Prescription — automated diagnosis with actionable suggestions

Typical use inside an exercise::


    with DLDiagnostics(model) as diag:
        diag.track_gradients()
        diag.track_activations()
        diag.track_dead_neurons()
        for epoch in range(epochs):
            for batch in dataloader:
                loss = train_step(model, batch)
                diag.record_batch(loss=loss.item(),
                                  lr=optimizer.param_groups[0]["lr"])
            diag.record_epoch(val_loss=evaluate(model, val_loader))
        diag.plot_training_dashboard().show()
        diag.report()

All DataFrames are Polars. All plots are Plotly. No matplotlib, no pandas.
"""

import logging
import math
from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import plotly.graph_objects as go
import polars as pl
import torch
import torch.nn as nn
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader


logger = logging.getLogger(__name__)

# Module names whose outputs are "dead-neuron sensitive" — we track fraction
# of zero outputs for these specifically.
_DEAD_NEURON_SENSITIVE: tuple[type[nn.Module], ...] = (
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
)

# Module names whose outputs we monitor for activation statistics.
_ACTIVATION_MONITORED: tuple[type[nn.Module], ...] = (
    nn.Linear,
    nn.Conv1d,
    nn.Conv2d,
    nn.Conv3d,
    nn.ReLU,
    nn.LeakyReLU,
    nn.GELU,
    nn.ELU,
    nn.SiLU,
    nn.Tanh,
    nn.Sigmoid,
    nn.BatchNorm1d,
    nn.BatchNorm2d,
    nn.LayerNorm,
)


@dataclass
class _HookHandles:
    """Container for registered hook handles so we can detach cleanly."""

    gradient: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    activation: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    dead_neuron: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)
    grad_cam: list[torch.utils.hooks.RemovableHandle] = field(default_factory=list)

    def all(self) -> list[torch.utils.hooks.RemovableHandle]:
        return self.gradient + self.activation + self.dead_neuron + self.grad_cam


class DLDiagnostics:
    """Clinical instrumentation for PyTorch training loops.

    Collects per-batch time series of gradient norms, activation statistics,
    dead-neuron fractions, and losses; exposes Plotly visualisations and an
    automated report that surfaces overfitting, vanishing gradients, and
    pathological dead-ReLU layers.

    Args:
        model: The ``nn.Module`` to instrument. The model is NOT modified;
            only forward/backward hooks are attached.
        dead_neuron_threshold: Fraction of zero outputs above which a layer
            is flagged as "dead" in :meth:`report`. Defaults to ``0.5``.
        window: Number of recent batches used for dead-neuron statistics.
            Defaults to ``64``.

    Example:
        >>> model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1))
        >>> with DLDiagnostics(model) as diag:
        ...     diag.track_gradients()
        ...     diag.track_activations()
        ...     # ... training loop ...
    """

    def __init__(
        self,
        model: nn.Module,
        *,
        dead_neuron_threshold: float = 0.5,
        window: int = 64,
    ) -> None:
        if not isinstance(model, nn.Module):
            raise TypeError(
                f"DLDiagnostics requires an nn.Module; got {type(model).__name__}"
            )
        if not 0.0 < dead_neuron_threshold < 1.0:
            raise ValueError("dead_neuron_threshold must be in (0, 1)")
        if window < 1:
            raise ValueError("window must be >= 1")

        self.model = model
        self.device = get_device()
        self.dead_neuron_threshold = dead_neuron_threshold
        self.window = window

        # Time series storage — lists of dicts, converted to Polars on demand.
        self._grad_log: list[dict[str, Any]] = []
        self._act_log: list[dict[str, Any]] = []
        self._dead_log: list[dict[str, Any]] = []
        self._batch_log: list[dict[str, Any]] = []
        self._epoch_log: list[dict[str, Any]] = []

        # Running per-layer firing masks for dead-neuron detection.
        # Key: layer name -> tensor of firing counts per neuron (1D).
        self._firing_counts: dict[str, torch.Tensor] = {}
        self._firing_samples: dict[str, int] = {}

        # Counters bound to hook closures so they share scope.
        self._batch_idx = 0
        self._epoch_idx = 0

        self._handles = _HookHandles()
        self._tracking = {"gradients": False, "activations": False, "dead": False}

        # Grad-CAM capture buffers (populated on demand).
        self._gradcam_activation: torch.Tensor | None = None
        self._gradcam_gradient: torch.Tensor | None = None

        logger.info(
            "dldiagnostics.init",
            extra={
                "model_class": type(model).__name__,
                "device": str(self.device),
                "window": window,
            },
        )

    # ── Context-manager support ────────────────────────────────────────────

    def __enter__(self) -> DLDiagnostics:
        return self

    def __exit__(self, exc_type, exc, tb) -> None:  # noqa: D401
        self.detach()

    def __del__(self) -> None:  # pragma: no cover - best-effort cleanup
        try:
            self.detach()
        except Exception:
            # Finalizers MUST NOT raise. Silent cleanup is the documented
            # contract for __del__ in rules/patterns.md.
            pass

    # ── Hook registration ──────────────────────────────────────────────────

    def track_gradients(self) -> DLDiagnostics:
        """Register backward hooks on every trainable parameter.

        Records the L2 norm of each parameter's gradient at every backward
        pass, keyed by parameter name.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["gradients"]:
            return self
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            handle = param.register_hook(self._make_grad_hook(name))
            self._handles.gradient.append(handle)
        self._tracking["gradients"] = True
        logger.info(
            "dldiagnostics.track_gradients",
            extra={"hooks_registered": len(self._handles.gradient)},
        )
        return self

    def track_activations(self) -> DLDiagnostics:
        """Register forward hooks on monitored submodules.

        Records mean/std/min/max/dead-fraction of activations per layer
        at every forward pass.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["activations"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _ACTIVATION_MONITORED):
                continue
            handle = module.register_forward_hook(self._make_act_hook(name))
            self._handles.activation.append(handle)
        self._tracking["activations"] = True
        logger.info(
            "dldiagnostics.track_activations",
            extra={"hooks_registered": len(self._handles.activation)},
        )
        return self

    def track_dead_neurons(self) -> DLDiagnostics:
        """Register forward hooks on ReLU-family layers to track dead neurons.

        A "neuron" here is a channel (Conv) or output unit (Linear). The
        rolling fraction of batches where that neuron output zero is tracked.

        Returns:
            ``self`` for chaining.
        """
        if self._tracking["dead"]:
            return self
        for name, module in self.model.named_modules():
            if name == "" or not isinstance(module, _DEAD_NEURON_SENSITIVE):
                continue
            handle = module.register_forward_hook(self._make_dead_hook(name))
            self._handles.dead_neuron.append(handle)
        self._tracking["dead"] = True
        logger.info(
            "dldiagnostics.track_dead_neurons",
            extra={"hooks_registered": len(self._handles.dead_neuron)},
        )
        return self

    def detach(self) -> None:
        """Remove ALL registered hooks and release references.

        Safe to call multiple times. Invoked automatically on context exit.
        """
        for handle in self._handles.all():
            try:
                handle.remove()
            except Exception:
                # Hook removal failures are benign (module may already be
                # gone). See rules/zero-tolerance.md Rule 3 carve-out for
                # cleanup paths.
                pass
        self._handles = _HookHandles()
        self._tracking = {k: False for k in self._tracking}
        self._gradcam_activation = None
        self._gradcam_gradient = None

    # ── Recording ─────────────────────────────────────────────────────────

    def record_batch(self, *, loss: float, lr: float | None = None) -> None:
        """Record per-batch scalar training signals.

        Args:
            loss: Training loss for the batch (post-backward).
            lr: Current learning rate (optional; read from optimizer).
        """
        if not math.isfinite(loss):
            logger.warning(
                "dldiagnostics.record_batch.nonfinite_loss",
                extra={"loss": str(loss), "batch": self._batch_idx},
            )
        self._batch_log.append(
            {
                "batch": self._batch_idx,
                "epoch": self._epoch_idx,
                "loss": float(loss),
                "lr": float(lr) if lr is not None else float("nan"),
            }
        )
        self._batch_idx += 1

    def record_epoch(
        self,
        *,
        val_loss: float | None = None,
        train_loss: float | None = None,
        **extra: float,
    ) -> None:
        """Record per-epoch summary metrics.

        Args:
            val_loss: Validation loss at epoch end.
            train_loss: Mean training loss for the epoch. If ``None``, it is
                computed from the batches in this epoch.
            **extra: Any additional scalar metrics to persist.
        """
        if train_loss is None:
            epoch_batches = [
                b for b in self._batch_log if b["epoch"] == self._epoch_idx
            ]
            if epoch_batches:
                train_loss = float(np.mean([b["loss"] for b in epoch_batches]))
        entry = {
            "epoch": self._epoch_idx,
            "train_loss": train_loss if train_loss is not None else float("nan"),
            "val_loss": val_loss if val_loss is not None else float("nan"),
            **{k: float(v) for k, v in extra.items()},
        }
        self._epoch_log.append(entry)
        self._epoch_idx += 1

    # ── Public DataFrame accessors ────────────────────────────────────────

    def gradients_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer gradient norms by batch."""
        if not self._grad_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "grad_norm": pl.Float64,
                    "grad_rms": pl.Float64,
                    "update_ratio": pl.Float64,
                }
            )
        return pl.DataFrame(self._grad_log)

    def activations_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-layer activation statistics by batch."""
        if not self._act_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "layer": pl.Utf8,
                    "act_kind": pl.Utf8,
                    "mean": pl.Float64,
                    "std": pl.Float64,
                    "min": pl.Float64,
                    "max": pl.Float64,
                    "dead_fraction": pl.Float64,
                    "inactivity_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(self._act_log)

    def dead_neurons_df(self) -> pl.DataFrame:
        """Polars DataFrame of current per-layer dead-neuron fractions."""
        rows: list[dict[str, Any]] = []
        for name, counts in self._firing_counts.items():
            n_samples = max(self._firing_samples.get(name, 1), 1)
            dead_mask = (counts / n_samples) < 1e-6
            rows.append(
                {
                    "layer": name,
                    "n_neurons": int(counts.numel()),
                    "n_dead": int(dead_mask.sum().item()),
                    "dead_fraction": float(dead_mask.float().mean().item()),
                }
            )
        if not rows:
            return pl.DataFrame(
                schema={
                    "layer": pl.Utf8,
                    "n_neurons": pl.Int64,
                    "n_dead": pl.Int64,
                    "dead_fraction": pl.Float64,
                }
            )
        return pl.DataFrame(rows)

    def batches_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-batch scalars (loss, lr)."""
        if not self._batch_log:
            return pl.DataFrame(
                schema={
                    "batch": pl.Int64,
                    "epoch": pl.Int64,
                    "loss": pl.Float64,
                    "lr": pl.Float64,
                }
            )
        return pl.DataFrame(self._batch_log)

    def epochs_df(self) -> pl.DataFrame:
        """Polars DataFrame of per-epoch summary metrics."""
        if not self._epoch_log:
            return pl.DataFrame(
                schema={
                    "epoch": pl.Int64,
                    "train_loss": pl.Float64,
                    "val_loss": pl.Float64,
                }
            )
        return pl.DataFrame(self._epoch_log)

    # ── Plots ─────────────────────────────────────────────────────────────

    def plot_loss_curves(self) -> go.Figure:
        """Loss stethoscope: train vs val curves with overfitting callout.

        Returns:
            A Plotly Figure with two traces (train / val) and annotations
            for detected overfitting epoch (if any).
        """
        epochs = self.epochs_df()
        batches = self.batches_df()
        fig = go.Figure()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train (per batch)",
                    line=dict(color="lightblue", width=1),
                    opacity=0.5,
                )
            )
        if epochs.height:
            fig.add_trace(
                go.Scatter(
                    x=epochs["epoch"].to_list(),
                    y=epochs["train_loss"].to_list(),
                    mode="lines+markers",
                    name="train (epoch mean)",
                    line=dict(color="steelblue", width=2),
                )
            )
            if epochs["val_loss"].is_not_null().any():
                fig.add_trace(
                    go.Scatter(
                        x=epochs["epoch"].to_list(),
                        y=epochs["val_loss"].to_list(),
                        mode="lines+markers",
                        name="val",
                        line=dict(color="firebrick", width=2),
                    )
                )
        overfit_epoch = self._detect_overfit_epoch()
        if overfit_epoch is not None:
            fig.add_vline(
                x=overfit_epoch,
                line=dict(color="orange", dash="dash"),
                annotation_text=f"overfitting suspected @ epoch {overfit_epoch}",
                annotation_position="top",
            )
        fig.update_layout(
            title="Loss Curves (Stethoscope)",
            xaxis_title="step",
            yaxis_title="loss",
            template="plotly_white",
            hovermode="x unified",
        )
        return fig

    def plot_gradient_flow(self) -> go.Figure:
        """Blood test: per-layer gradient norm over time.

        One trace per tracked parameter. Layers whose gradient norm collapses
        toward zero are the vanishing-gradient culprits.
        """
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Gradient Flow (Blood Test) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["grad_norm"].to_list(),
                    mode="lines",
                    name=layer,
                    hovertemplate=f"{layer}<br>batch=%{{x}}<br>‖∇‖=%{{y:.3e}}",
                )
            )
        fig.update_layout(
            title="Gradient Flow by Layer (Blood Test)",
            xaxis_title="batch",
            yaxis_title="gradient L2 norm",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_activation_stats(self) -> go.Figure:
        """X-Ray: activation mean ± std per layer over time."""
        df = self.activations_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Activation Statistics (X-Ray) — no data",
                template="plotly_white",
            )
            return fig
        for layer in df["layer"].unique().to_list():
            sub = df.filter(pl.col("layer") == layer)
            fig.add_trace(
                go.Scatter(
                    x=sub["batch"].to_list(),
                    y=sub["mean"].to_list(),
                    mode="lines",
                    name=f"{layer} — mean",
                    hovertemplate=(
                        f"{layer}<br>batch=%{{x}}<br>mean=%{{y:.3f}}<br>"
                        "std=%{customdata:.3f}"
                    ),
                    customdata=sub["std"].to_list(),
                )
            )
        fig.update_layout(
            title="Activation Mean per Layer (X-Ray)",
            xaxis_title="batch",
            yaxis_title="activation mean",
            template="plotly_white",
        )
        return fig

    def plot_dead_neurons(self) -> go.Figure:
        """X-Ray: fraction of dead neurons per ReLU-family layer."""
        df = self.dead_neurons_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(
                title="Dead Neurons (X-Ray) — no ReLU-family layers tracked",
                template="plotly_white",
            )
            return fig
        colors = [
            "firebrick" if frac > self.dead_neuron_threshold else "steelblue"
            for frac in df["dead_fraction"].to_list()
        ]
        fig.add_trace(
            go.Bar(
                x=df["layer"].to_list(),
                y=df["dead_fraction"].to_list(),
                marker_color=colors,
                text=[
                    f"{int(n)}/{int(t)}" for n, t in zip(df["n_dead"], df["n_neurons"])
                ],
                textposition="outside",
            )
        )
        fig.add_hline(
            y=self.dead_neuron_threshold,
            line=dict(color="orange", dash="dash"),
            annotation_text=f"alert threshold ({self.dead_neuron_threshold:.0%})",
        )
        fig.update_layout(
            title="Dead Neuron Fraction by Layer (X-Ray)",
            xaxis_title="layer",
            yaxis_title="fraction dead",
            yaxis=dict(range=[0, 1]),
            template="plotly_white",
            showlegend=False,
        )
        return fig

    def plot_training_dashboard(self) -> go.Figure:
        """Vital signs: 2x2 dashboard (loss, grad flow, activations, LR)."""
        fig = make_subplots(
            rows=2,
            cols=2,
            subplot_titles=(
                "Loss (Stethoscope)",
                "Gradient Flow (Blood Test)",
                "Activation Mean (X-Ray)",
                "Learning Rate",
            ),
        )

        # (1,1) Loss
        epochs = self.epochs_df()
        batches = self.batches_df()
        if batches.height:
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["loss"].to_list(),
                    mode="lines",
                    name="train loss",
                    line=dict(color="steelblue", width=1),
                ),
                row=1,
                col=1,
            )
        if epochs.height and epochs["val_loss"].is_not_null().any():
            # Place val loss at the last batch of each epoch for alignment.
            val_x = []
            for ep in epochs["epoch"].to_list():
                ep_batches = batches.filter(pl.col("epoch") == ep)
                val_x.append(
                    int(ep_batches["batch"].max()) if ep_batches.height else ep
                )
            fig.add_trace(
                go.Scatter(
                    x=val_x,
                    y=epochs["val_loss"].to_list(),
                    mode="lines+markers",
                    name="val loss",
                    line=dict(color="firebrick"),
                ),
                row=1,
                col=1,
            )

        # (1,2) Gradient flow
        gdf = self.gradients_df()
        if gdf.height:
            for layer in gdf["layer"].unique().to_list():
                sub = gdf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["grad_norm"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=1,
                    col=2,
                )
            fig.update_yaxes(type="log", row=1, col=2)

        # (2,1) Activation mean
        adf = self.activations_df()
        if adf.height:
            for layer in adf["layer"].unique().to_list():
                sub = adf.filter(pl.col("layer") == layer)
                fig.add_trace(
                    go.Scatter(
                        x=sub["batch"].to_list(),
                        y=sub["mean"].to_list(),
                        mode="lines",
                        name=layer,
                        showlegend=False,
                    ),
                    row=2,
                    col=1,
                )

        # (2,2) LR
        if batches.height and batches["lr"].is_not_null().any():
            fig.add_trace(
                go.Scatter(
                    x=batches["batch"].to_list(),
                    y=batches["lr"].to_list(),
                    mode="lines",
                    name="lr",
                    line=dict(color="darkgreen"),
                    showlegend=False,
                ),
                row=2,
                col=2,
            )

        fig.update_layout(
            title="Training Dashboard (Vital Signs)",
            template="plotly_white",
            height=720,
        )
        return fig

    def plot_lr_vs_loss(self) -> go.Figure:
        """Plot LR vs loss (useful after an :meth:`lr_range_test`)."""
        df = self.batches_df()
        fig = go.Figure()
        if df.height == 0 or df["lr"].is_null().all():
            fig.update_layout(title="LR vs Loss — no data", template="plotly_white")
            return fig
        fig.add_trace(
            go.Scatter(
                x=df["lr"].to_list(),
                y=df["loss"].to_list(),
                mode="lines",
                line=dict(color="steelblue"),
            )
        )
        fig.update_layout(
            title="Learning Rate vs Loss",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        return fig

    def plot_weight_distributions(self) -> go.Figure:
        """Histogram of weight values, one trace per parameter tensor."""
        fig = go.Figure()
        for name, param in self.model.named_parameters():
            if not param.requires_grad or param.ndim == 0:
                continue
            values = param.detach().cpu().flatten().numpy()
            fig.add_trace(go.Histogram(x=values, name=name, opacity=0.6))
        fig.update_layout(
            title="Weight Distributions",
            xaxis_title="value",
            yaxis_title="count",
            barmode="overlay",
            template="plotly_white",
        )
        return fig

    def plot_gradient_norms(self) -> go.Figure:
        """Mean gradient norm per layer across the run (bar chart)."""
        df = self.gradients_df()
        fig = go.Figure()
        if df.height == 0:
            fig.update_layout(title="Gradient Norms — no data", template="plotly_white")
            return fig
        summary = df.group_by("layer").agg(
            pl.col("grad_norm").mean().alias("mean_grad")
        )
        summary = summary.sort("mean_grad")
        fig.add_trace(
            go.Bar(
                x=summary["layer"].to_list(),
                y=summary["mean_grad"].to_list(),
                marker_color="steelblue",
            )
        )
        fig.update_layout(
            title="Mean Gradient Norm per Layer",
            xaxis_title="layer",
            yaxis_title="mean ‖∇‖",
            yaxis_type="log",
            template="plotly_white",
        )
        return fig

    # ── Automated report ──────────────────────────────────────────────────

    def report(self) -> dict[str, Any]:
        """Print a human-readable diagnosis and return findings as a dict.

        Findings covered:
            * Gradient flow (vanishing / exploding / healthy)
            * Dead neurons (per-layer ReLU-family)
            * Loss trend (overfitting, underfitting, instability)

        Returns:
            A dict with keys ``gradient_flow``, ``dead_neurons``, ``loss_trend``
            each mapping to a dict with ``severity`` and ``message``.
        """
        findings: dict[str, Any] = {}

        # 1. Gradient flow — uses SCALE-INVARIANT per-element RMS (grad_rms)
        # and update_ratio (‖∇W‖/‖W‖), matching slide 5F thresholds.
        gdf = self.gradients_df()
        if gdf.height and "grad_rms" in gdf.columns:
            stats = gdf.group_by("layer").agg(
                pl.col("grad_rms").mean().alias("rms"),
                pl.col("update_ratio").mean().alias("ur"),
            )
            min_rms_raw = stats["rms"].min()
            max_rms_raw = stats["rms"].max()
            min_ur_raw = stats["ur"].min()
            max_ur_raw = stats["ur"].max()
            min_rms = (
                float(min_rms_raw) if isinstance(min_rms_raw, (int, float)) else None
            )
            max_rms = (
                float(max_rms_raw) if isinstance(max_rms_raw, (int, float)) else None
            )
            min_ur = float(min_ur_raw) if isinstance(min_ur_raw, (int, float)) else 0.0
            max_ur = float(max_ur_raw) if isinstance(max_ur_raw, (int, float)) else 0.0
            if min_rms is None or max_rms is None or min_rms == 0:
                severity = "UNKNOWN"
                message = "Insufficient gradient data."
            elif min_rms < 1e-5 or min_ur < 1e-4:
                # Vanishing: RMS < 1e-5 OR update ratio < 1e-4 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms").row(0, named=True)["layer"]
                    if min_rms < 1e-5
                    else stats.sort("ur").row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Vanishing gradients at '{worst_layer}' — "
                    f"min RMS = {min_rms:.2e}, min update_ratio = {min_ur:.2e}. "
                    "Fix: verify pre-norm layout (LayerNorm/RMSNorm before block), "
                    "add residual connections, switch to GELU/SiLU, or use Kaiming init."
                )
            elif max_rms > 1e-2 or max_ur > 0.1:
                # Exploding: RMS > 1e-2 OR update ratio > 0.1 (matches slide 5F).
                worst_layer = (
                    stats.sort("rms", descending=True).row(0, named=True)["layer"]
                    if max_rms > 1e-2
                    else stats.sort("ur", descending=True).row(0, named=True)["layer"]
                )
                severity = "CRITICAL"
                message = (
                    f"Exploding gradients at '{worst_layer}' — "
                    f"max RMS = {max_rms:.2e}, max update_ratio = {max_ur:.2e}. "
                    "Fix: add gradient clipping (or adaptive: ZClip/AGC), reduce LR, "
                    "verify initialization (Kaiming for ReLU, Xavier for Tanh)."
                )
            elif max_rms / max(min_rms, 1e-12) > 1e3:
                severity = "WARNING"
                message = (
                    f"Large RMS spread across layers (max/min = "
                    f"{max_rms / min_rms:.1e}). Deep layers may be learning unevenly."
                )
            else:
                severity = "HEALTHY"
                message = (
                    f"Gradient flow OK (RMS range {min_rms:.2e}–{max_rms:.2e}, "
                    f"update_ratio range {min_ur:.2e}–{max_ur:.2e})."
                )
            findings["gradient_flow"] = {"severity": severity, "message": message}
        elif gdf.height:
            # Legacy path for dataframes without the new columns.
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": (
                    "grad_rms field missing — re-run with the current library "
                    "version to get scale-invariant diagnosis."
                ),
            }
        else:
            findings["gradient_flow"] = {
                "severity": "UNKNOWN",
                "message": "No gradient tracking enabled — call track_gradients().",
            }

        # 2. Dead neurons / saturation — uses ACTIVATION-TYPE-AWARE
        # inactivity_fraction from _act_log (ReLU: |x|<eps; Tanh: |x|>0.99;
        # Sigmoid: |x|>0.99 or |x|<0.01). This catches near-dead ReLU channels
        # that the legacy exact-zero check misses post-BatchNorm.
        adf = self.activations_df()
        if adf.height and "inactivity_fraction" in adf.columns:
            # Aggregate mean inactivity per layer (averaged across batches).
            agg = (
                adf.filter(pl.col("act_kind") != "other")
                .group_by("layer")
                .agg(
                    pl.col("inactivity_fraction").mean().alias("mean_inactive"),
                    pl.col("act_kind").first().alias("kind"),
                )
                .sort("mean_inactive", descending=True)
            )
            if agg.height:
                worst = agg.row(0, named=True)
                thr = self.dead_neuron_threshold
                if worst["mean_inactive"] > thr:
                    kind = worst["kind"]
                    if kind == "relu":
                        label = "dead neurons"
                        fix = "Switch to GELU/LeakyReLU or re-initialise with Kaiming."
                    elif kind == "tanh":
                        label = "saturated (|x|>0.99)"
                        fix = "Reduce LR, add LayerNorm before, or switch to GELU."
                    elif kind == "sigmoid":
                        label = "saturated (|x|>0.99 or |x|<0.01)"
                        fix = "Reduce LR, add BN/LN, or switch to softmax+CE if classification."
                    else:
                        label = "inactive"
                        fix = "Investigate activation distribution."
                    findings["dead_neurons"] = {
                        "severity": "WARNING",
                        "message": (
                            f"'{worst['layer']}' ({kind}): "
                            f"{worst['mean_inactive']:.0%} {label}. {fix}"
                        ),
                    }
                else:
                    findings["dead_neurons"] = {
                        "severity": "HEALTHY",
                        "message": (
                            f"All {agg.height} activation layers healthy "
                            f"(worst: {worst['layer']} at "
                            f"{worst['mean_inactive']:.0%} inactive, below "
                            f"{thr:.0%} threshold)."
                        ),
                    }
            else:
                findings["dead_neurons"] = {
                    "severity": "UNKNOWN",
                    "message": "No activation layers tracked — call track_activations().",
                }
        else:
            findings["dead_neurons"] = {
                "severity": "UNKNOWN",
                "message": "No activation tracking enabled — call track_activations().",
            }

        # 3. Loss trend
        edf = self.epochs_df()
        if edf.height >= 3 and edf["val_loss"].is_not_null().any():
            overfit = self._detect_overfit_epoch()
            train_trend = self._slope(edf["train_loss"].to_list())
            if overfit is not None:
                severity = "WARNING"
                message = (
                    f"Overfitting suspected at epoch {overfit} "
                    "(val loss rising while train loss falls). "
                    "Consider dropout, weight decay, or early stopping."
                )
            elif train_trend > -1e-4 and edf.height >= 5:
                severity = "WARNING"
                message = (
                    f"Underfitting — train loss slope {train_trend:.2e}/epoch. "
                    "Consider a larger model, more epochs, or higher LR."
                )
            else:
                severity = "HEALTHY"
                message = f"Loss converging (train slope {train_trend:.2e}/epoch)."
            findings["loss_trend"] = {"severity": severity, "message": message}
        else:
            findings["loss_trend"] = {
                "severity": "UNKNOWN",
                "message": "Need at least 3 epochs with val_loss for trend analysis.",
            }

        # Human-readable summary
        print("\n" + "═" * 64)
        print("  DL Diagnostics Report — Prescription Pad")
        print("═" * 64)
        icons = {
            "HEALTHY": "✓",
            "WARNING": "!",
            "CRITICAL": "X",
            "UNKNOWN": "?",
        }
        titles = {
            "gradient_flow": "Gradient flow",
            "dead_neurons": "Dead neurons ",
            "loss_trend": "Loss trend   ",
        }
        for key, label in titles.items():
            f = findings[key]
            print(
                f"  [{icons[f['severity']]}] {label} ({f['severity']}): {f['message']}"
            )
        print("═" * 64 + "\n")
        return findings

    # ── Grad-CAM ──────────────────────────────────────────────────────────

    def grad_cam(
        self,
        input_tensor: torch.Tensor,
        target_class: int,
        layer_name: str,
    ) -> torch.Tensor:
        """Compute a Grad-CAM heatmap for ``target_class`` from ``layer_name``.

        Args:
            input_tensor: Input batch ``(N, C, H, W)`` or ``(N, C, L)``.
            target_class: Target class index to explain.
            layer_name: ``model.named_modules()`` key of the conv layer to
                attribute. Usually the last convolutional layer.

        Returns:
            Heatmap tensor with shape matching the spatial dims of the tracked
            layer's output (``(N, H', W')`` for 2D inputs).

        Raises:
            ValueError: If ``layer_name`` is not found in the model.
        """
        target_module: nn.Module | None = None
        for name, module in self.model.named_modules():
            if name == layer_name:
                target_module = module
                break
        if target_module is None:
            raise ValueError(
                f"Layer '{layer_name}' not found in model. "
                f"Available: {[n for n, _ in self.model.named_modules() if n][:10]}..."
            )

        self._gradcam_activation = None
        self._gradcam_gradient = None

        def fwd_hook(_mod: nn.Module, _inp: Any, out: torch.Tensor) -> None:
            self._gradcam_activation = out.detach()

        def bwd_hook(_mod: nn.Module, _gi: Any, go: Any) -> None:
            # go is a tuple — first element is d(output)/d(module-output)
            self._gradcam_gradient = go[0].detach()

        h1 = target_module.register_forward_hook(fwd_hook)
        h2 = target_module.register_full_backward_hook(bwd_hook)
        self._handles.grad_cam.extend([h1, h2])

        # Preserve caller's train/eval state — critical for mid-training use.
        was_training = self.model.training

        try:
            self.model.eval()
            inp = input_tensor.to(self.device)
            logits = self.model(inp)
            if logits.ndim != 2:
                raise ValueError(
                    f"grad_cam expects classification logits (N, C); got {logits.shape}"
                )
            self.model.zero_grad(set_to_none=True)
            one_hot = torch.zeros_like(logits)
            one_hot[:, target_class] = 1.0
            logits.backward(gradient=one_hot, retain_graph=False)

            if self._gradcam_activation is None or self._gradcam_gradient is None:
                raise RuntimeError(
                    "Grad-CAM hooks did not fire — layer may be unreachable "
                    "from the forward path."
                )
            activations = self._gradcam_activation  # (N, K, ...)
            gradients = self._gradcam_gradient  # (N, K, ...)
            # Global-average-pool gradients over spatial dims to get per-channel weights.
            spatial_dims = tuple(range(2, gradients.ndim))
            weights = gradients.mean(dim=spatial_dims, keepdim=True)  # (N, K, 1, ...)
            cam = (weights * activations).sum(dim=1)  # (N, ...)
            cam = torch.relu(cam)
            # Normalise per-sample to [0, 1].
            flat = cam.flatten(start_dim=1)
            mins = flat.min(dim=1, keepdim=True).values
            maxs = flat.max(dim=1, keepdim=True).values
            denom = (maxs - mins).clamp(min=1e-8)
            flat = (flat - mins) / denom
            cam = flat.view_as(cam)
            return cam.cpu()
        finally:
            # Restore caller's train/eval state BEFORE removing hooks, so
            # downstream errors in hook cleanup don't leave model in eval mode.
            if was_training:
                self.model.train()
            h1.remove()
            h2.remove()
            # Remove from bookkeeping too so detach() stays idempotent.
            self._handles.grad_cam = [
                h for h in self._handles.grad_cam if h is not h1 and h is not h2
            ]
            self._gradcam_activation = None
            self._gradcam_gradient = None

    # ── LR range test (static) ────────────────────────────────────────────

    @staticmethod
    def lr_range_test(
        model: nn.Module,
        dataloader: DataLoader,
        *,
        loss_fn: nn.Module | None = None,
        optimizer_cls: type[torch.optim.Optimizer] = torch.optim.SGD,
        lr_min: float = 1e-7,
        lr_max: float = 10.0,
        steps: int = 200,
        device: torch.device | None = None,
        batch_adapter: Callable[[Any], tuple[torch.Tensor, torch.Tensor]] | None = None,
        safety_divisor: float = 10.0,
    ) -> dict[str, Any]:
        """Leslie Smith LR range test (with fastai-style safe-LR recipe).

        Trains for ``steps`` batches while exponentially increasing LR from
        ``lr_min`` to ``lr_max``. Smooths the loss curve with EMA (beta=0.98)
        before finding the steepest-descent point — this matches fastai's
        ``lr_find()`` and avoids picking a single lucky batch in the tail.

        **Critical**: returns BOTH ``min_loss_lr`` (steepest descent) AND
        ``safe_lr = min_loss_lr / safety_divisor`` (default 10). Use ``safe_lr``
        in your optimizer — ``min_loss_lr`` is the edge of instability.

        The model's weights are saved before the test and **restored** on exit
        (via state_dict deepcopy), so calling this does not corrupt your model.

        Args:
            model: The model to probe. Weights are restored after return.
            dataloader: Any DataLoader. By default yields ``(inputs, targets)``
                tuples; pass ``batch_adapter`` for HF/SSL batch formats.
            loss_fn: Loss function (REQUIRED — no silent default).
            optimizer_cls: Optimizer class (default SGD).
            lr_min, lr_max, steps: Sweep configuration.
            device: Override compute device (default: model's current device).
            batch_adapter: Callable ``batch -> (x, y)``. Default handles
                tuple/list; pass a lambda for dict-shaped batches (e.g. HF).
            safety_divisor: Divide steepest-descent LR by this to get safe LR
                (fastai default: 10).

        Returns:
            ``{"safe_lr": float,              # use this in your optimizer
                "min_loss_lr": float,          # steepest descent (edge of instability)
                "divergence_lr": float,        # where smoothed loss > 4× min
                "lrs": [...], "losses": [...], "losses_smooth": [...],
                "figure": go.Figure}``
        """
        if steps < 2:
            raise ValueError("steps must be >= 2")
        if not 0 < lr_min < lr_max:
            raise ValueError("require 0 < lr_min < lr_max")
        if loss_fn is None:
            raise ValueError(
                "loss_fn is required — no silent default. "
                "Pass nn.CrossEntropyLoss() for classification or nn.MSELoss() for regression."
            )

        import copy as _copy

        # Device: honor model's current device unless overridden.
        dev = device or next(model.parameters()).device
        if device is not None:
            model = model.to(dev)

        # Save weights for restoration.
        saved_state = _copy.deepcopy(model.state_dict())

        # Default batch adapter handles tuple/list; raises loudly on dicts.
        def _default_adapter(batch: Any) -> tuple[torch.Tensor, torch.Tensor]:
            if isinstance(batch, (tuple, list)) and len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(
                "lr_range_test got a non-(x,y) batch. Pass batch_adapter=... "
                "for HuggingFace-style dict batches or SSL single-tensor batches."
            )

        adapter = batch_adapter or _default_adapter

        try:
            optimizer = optimizer_cls(model.parameters(), lr=lr_min)
            gamma = (lr_max / lr_min) ** (1.0 / (steps - 1))
            lrs: list[float] = []
            losses: list[float] = []
            running_min = float("inf")  # O(1) running min, not O(n)
            model.train()
            data_iter = iter(dataloader)
            for step in range(steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(dataloader)
                    batch = next(data_iter)
                x, y = adapter(batch)
                x, y = x.to(dev), y.to(dev)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr_min * (gamma**step)
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                optimizer.step()
                cur_lr = optimizer.param_groups[0]["lr"]
                cur_loss = float(loss.item())
                lrs.append(cur_lr)
                losses.append(cur_loss)
                if cur_loss < running_min:
                    running_min = cur_loss
                if not math.isfinite(cur_loss) or cur_loss > 10 * running_min:
                    logger.info(
                        "dldiagnostics.lr_range_test.diverged",
                        extra={"step": step, "lr": cur_lr, "loss": cur_loss},
                    )
                    break
        finally:
            # Always restore weights — no silent corruption.
            model.load_state_dict(saved_state)

        # EMA-smoothed losses (fastai convention, beta=0.98).
        beta = 0.98
        losses_smooth: list[float] = []
        ema = 0.0
        for i, ell in enumerate(losses):
            ema = beta * ema + (1 - beta) * ell
            # Bias-correct the EMA.
            losses_smooth.append(ema / (1 - beta ** (i + 1)))

        # min_loss_lr = LR at the steepest-descent point of SMOOTHED loss.
        min_loss_lr = lr_min
        divergence_lr = lr_max
        if len(losses_smooth) >= 3:
            dl = np.diff(np.array(losses_smooth))
            idx = int(np.argmin(dl))
            min_loss_lr = lrs[idx]
            # Divergence = first point where smoothed loss > 4× its minimum.
            min_smooth = min(losses_smooth)
            for i, s in enumerate(losses_smooth):
                if s > 4 * min_smooth and i > idx:
                    divergence_lr = lrs[i]
                    break

        safe_lr = min_loss_lr / safety_divisor

        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses,
                mode="lines",
                name="raw loss",
                line=dict(color="lightgray"),
            )
        )
        fig.add_trace(
            go.Scatter(
                x=lrs,
                y=losses_smooth,
                mode="lines",
                name="smoothed",
                line=dict(color="#0D9488", width=2),
            )
        )
        fig.add_vline(
            x=safe_lr,
            line=dict(color="#22C55E", dash="dash"),
            annotation_text=f"safe_lr = {safe_lr:.2e}",
        )
        fig.add_vline(
            x=min_loss_lr,
            line=dict(color="#F59E0B", dash="dot"),
            annotation_text=f"min_loss_lr = {min_loss_lr:.2e}",
        )
        fig.update_layout(
            title="LR Range Test (smoothed) — use safe_lr, not min_loss_lr",
            xaxis_title="learning rate",
            yaxis_title="loss",
            xaxis_type="log",
            template="plotly_white",
        )
        logger.info(
            "dldiagnostics.lr_range_test.ok",
            extra={
                "safe_lr": safe_lr,
                "min_loss_lr": min_loss_lr,
                "divergence_lr": divergence_lr,
                "steps_run": len(lrs),
            },
        )
        return {
            "safe_lr": safe_lr,
            "min_loss_lr": min_loss_lr,
            "divergence_lr": divergence_lr,
            "suggested_lr": safe_lr,  # backwards-compat alias
            "lrs": lrs,
            "losses": losses,
            "losses_smooth": losses_smooth,
            "figure": fig,
        }

    # ── Hook factories (internal) ─────────────────────────────────────────

    def _make_grad_hook(self, name: str):
        """Scale-invariant gradient tracking.

        Records three metrics per layer per batch:
          - grad_norm: raw L2 norm (preserved for backwards compatibility)
          - grad_rms: per-element RMS = ‖∇‖ / sqrt(numel) — scale-invariant,
            comparable across layers of any size. This is what LLM training
            dashboards (W&B, TensorBoard) actually display.
          - update_ratio: ‖∇W‖ / ‖W‖ — the "effective LR" per layer.

        Casts to fp32 before reduction so BF16/FP16 gradients don't silently
        produce inf/NaN.
        """
        # Look up the parameter tensor for update-ratio computation.
        try:
            param = dict(self.model.named_parameters())[name]
        except KeyError:
            param = None

        def _hook(grad: torch.Tensor) -> None:
            try:
                # Handle sparse gradients (e.g. nn.Embedding(sparse=True)).
                g = grad.coalesce().values() if grad.is_sparse else grad
                # Cast to fp32 to avoid BF16/FP16 inf.
                g_f32 = g.detach().float()
                l2 = float(g_f32.norm(p=2).item())
                numel = max(g_f32.numel(), 1)
                rms = l2 / (numel**0.5)
                # Update ratio — use detached param weight norm.
                if param is not None:
                    p_norm = float(param.detach().float().norm(p=2).item())
                    upd_ratio = l2 / max(p_norm, 1e-12)
                else:
                    upd_ratio = 0.0
            except Exception as exc:  # pragma: no cover - defensive
                logger.warning(
                    "dldiagnostics.grad_hook.error",
                    extra={"layer": name, "error": str(exc)},
                )
                return
            self._grad_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "grad_norm": l2,  # preserved for compatibility
                    "grad_rms": rms,  # scale-invariant
                    "update_ratio": upd_ratio,  # ‖∇W‖ / ‖W‖
                }
            )

        return _hook

    def _make_act_hook(self, name: str):
        """Activation statistics. Casts to fp32 to avoid BF16/FP16 overflow.

        The `inactivity_fraction` field measures activation-type-appropriate
        pathology:
          - ReLU / GELU / SiLU: fraction with |x| < 1e-6 (dead neurons)
          - Tanh: fraction with |x| > 0.99 (saturated)
          - Sigmoid: fraction with |x| > 0.99 OR |x| < 0.01 (saturated)
          - Others: 0 (no pathology definition)

        The legacy `dead_fraction` field (exact-zero count) is preserved for
        backwards compatibility but is only meaningful for ReLU.
        """
        # Determine activation type from module class name for dispatching.
        act_kind = self._classify_activation_module(name)

        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            # Cast to fp32 for numerical safety (BF16/FP16 overflow on .std()).
            out = output.detach().float()
            try:
                mean = float(out.mean().item())
                std = float(out.std().item()) if out.numel() > 1 else 0.0
                mn = float(out.min().item())
                mx = float(out.max().item())
                legacy_dead = float((out == 0).float().mean().item())
                # Activation-aware inactivity/saturation metric.
                if act_kind == "relu":
                    inactivity = float((out.abs() < 1e-6).float().mean().item())
                elif act_kind == "tanh":
                    inactivity = float((out.abs() > 0.99).float().mean().item())
                elif act_kind == "sigmoid":
                    inactivity = float(
                        ((out > 0.99) | (out < 0.01)).float().mean().item()
                    )
                else:
                    inactivity = 0.0
            except RuntimeError:
                # Non-numeric tensor (e.g. mixed dtype). Skip silently.
                return
            # Guard against NaN/inf leaking into logs.
            for val_name, val in (("mean", mean), ("std", std)):
                if val != val or val in (float("inf"), float("-inf")):
                    logger.warning(
                        "dldiagnostics.act_hook.nonfinite",
                        extra={"layer": name, "field": val_name},
                    )
                    return
            self._act_log.append(
                {
                    "batch": self._batch_idx,
                    "layer": name,
                    "act_kind": act_kind,
                    "mean": mean,
                    "std": std,
                    "min": mn,
                    "max": mx,
                    "dead_fraction": legacy_dead,
                    "inactivity_fraction": inactivity,
                }
            )

        return _hook

    def _classify_activation_module(self, name: str) -> str:
        """Return one of 'relu', 'tanh', 'sigmoid', 'other' for a module name."""
        try:
            mod = dict(self.model.named_modules())[name]
        except KeyError:
            return "other"
        cls = type(mod).__name__.lower()
        if any(k in cls for k in ("relu", "gelu", "silu", "swish", "elu")):
            return "relu"
        if "tanh" in cls:
            return "tanh"
        if "sigmoid" in cls:
            return "sigmoid"
        return "other"

    def _make_dead_hook(self, name: str):
        def _hook(_module: nn.Module, _inp: Any, output: torch.Tensor) -> None:
            if not isinstance(output, torch.Tensor) or output.numel() == 0:
                return
            out = output.detach()
            # Collapse all non-channel dims. Convention: channel dim = 1 for
            # Conv outputs (N, C, ...); for Linear, output is (N, F) — here
            # we treat dim 1 as "neurons" which matches both shapes.
            if out.ndim < 2:
                return
            reduce_dims = tuple(d for d in range(out.ndim) if d != 1)
            fired = (out != 0).any(dim=reduce_dims).float().cpu()
            if name not in self._firing_counts:
                self._firing_counts[name] = torch.zeros_like(fired)
                self._firing_samples[name] = 0
            self._firing_counts[name] += fired
            self._firing_samples[name] += 1
            # Keep memory bounded by decaying old counts when window exceeded.
            if self._firing_samples[name] > self.window:
                scale = self.window / self._firing_samples[name]
                self._firing_counts[name] = self._firing_counts[name] * scale
                self._firing_samples[name] = self.window

        return _hook

    # ── Internal analysis helpers ─────────────────────────────────────────

    @staticmethod
    def _slope(series: list[float]) -> float:
        """Least-squares slope of a 1D series vs its index (ignores NaN)."""
        xs = np.arange(len(series), dtype=float)
        ys = np.asarray(series, dtype=float)
        mask = np.isfinite(ys)
        if mask.sum() < 2:
            return 0.0
        xs, ys = xs[mask], ys[mask]
        return float(np.polyfit(xs, ys, 1)[0])

    def _detect_overfit_epoch(self) -> int | None:
        """Return epoch index where val loss starts rising while train falls."""
        edf = self.epochs_df()
        if edf.height < 3:
            return None
        train = edf["train_loss"].to_list()
        val = edf["val_loss"].to_list()
        for i in range(2, len(val)):
            v_now, v_prev = val[i], val[i - 1]
            t_now, t_prev = train[i], train[i - 1]
            if any(
                x is None or not math.isfinite(x)
                for x in (v_now, v_prev, t_now, t_prev)
            ):
                continue
            if v_now > v_prev and t_now < t_prev:
                # Confirm trend persists one step back if available.
                return i
        return None


# ════════════════════════════════════════════════════════════════════════
# Standalone diagnostic checkpoint — for use AFTER a training loop
# ════════════════════════════════════════════════════════════════════════


def run_diagnostic_checkpoint(
    model: nn.Module,
    dataloader: DataLoader,
    loss_fn: Callable[..., Any],
    *,
    title: str = "Model",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    batch_adapter: Callable[[Any], tuple[torch.Tensor, ...]] | None = None,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Run a short instrumented diagnostic pass on a TRAINED model.

    Use this between the Train and Visualise phases of an exercise. It
    attaches the four diagnostic instruments (gradients, activations,
    dead-neurons, scalar history) to the model, runs ``n_batches`` of
    forward-backward passes to populate the history, replays any
    epoch-level losses you collected during real training, and prints the
    Prescription Pad with auto-diagnosis.

    The model's weights are NOT updated — gradients are computed but no
    optimizer step is taken. The model's train/eval state is preserved.

    Args:
        model: A trained ``nn.Module``.
        dataloader: Any DataLoader whose batches the loss function accepts.
        loss_fn: Callable. The default contract is
            ``loss_fn(model, batch) -> (scalar_loss, extras_dict)`` matching
            the M5 exercise convention. Pass ``batch_adapter`` to wrap a
            different signature.
        title: Human-readable name printed in the dashboard title.
        n_batches: How many batches to instrument (default 8 — enough for
            a stable mean per layer without slowing the exercise down).
        train_losses: Optional list of per-epoch train losses captured
            during the actual training run. Replayed into the dashboard's
            stethoscope view so students see the real loss trajectory, not
            just the diagnostic-pass losses.
        val_losses: Optional list of per-epoch validation losses, same
            length as ``train_losses``.
        show: If ``True``, calls ``.show()`` on the dashboard figure.
        batch_adapter: Optional ``batch -> (loss_fn_args...)`` translator
            for non-standard batch shapes.

    Returns:
        ``(diag, findings)`` so the caller can inspect the DataFrames and
        the Prescription Pad findings dict.
    """
    if not isinstance(model, nn.Module):
        raise TypeError("run_diagnostic_checkpoint requires an nn.Module")
    if n_batches < 1:
        raise ValueError("n_batches must be >= 1")

    diag = DLDiagnostics(model)
    diag.track_gradients().track_activations().track_dead_neurons()

    # Replay real training history into the dashboard if provided.
    if train_losses or val_losses:
        n_epochs = max(len(train_losses or []), len(val_losses or []))
        for i in range(n_epochs):
            tl = (
                float(train_losses[i])
                if train_losses and i < len(train_losses)
                else None
            )
            vl = float(val_losses[i]) if val_losses and i < len(val_losses) else None
            # Synthesise one batch entry per epoch so the per-batch stethoscope
            # has data to plot — students still see the real epoch losses.
            if tl is not None:
                diag.record_batch(loss=tl)
            diag.record_epoch(train_loss=tl, val_loss=vl)

    was_training = model.training
    try:
        model.train()  # Enable gradients + activation hooks.
        seen = 0
        for batch in dataloader:
            if seen >= n_batches:
                break
            try:
                if batch_adapter is not None:
                    args = batch_adapter(batch)
                    loss_out = loss_fn(model, *args)
                else:
                    loss_out = loss_fn(model, batch)
                # Convention: M5 loss_fn returns (loss, extras_dict). Allow
                # either a bare tensor or a tuple.
                if isinstance(loss_out, tuple):
                    loss = loss_out[0]
                else:
                    loss = loss_out
                if not isinstance(loss, torch.Tensor):
                    raise TypeError(
                        f"loss_fn returned {type(loss).__name__}; expected Tensor"
                    )
                model.zero_grad(set_to_none=True)
                loss.backward()
                # NOTE: no optimizer.step() — diagnostic pass is read-only.
                diag.record_batch(loss=float(loss.item()))
            except Exception as exc:  # pragma: no cover - student loop variability
                logger.warning(
                    "dldiagnostics.checkpoint.batch_skipped",
                    extra={"batch_idx": seen, "error": str(exc)},
                )
            seen += 1
    finally:
        if not was_training:
            model.eval()

    # Render the dashboard and the Prescription Pad.
    fig = diag.plot_training_dashboard()
    fig.update_layout(title=f"{title} — Diagnostic Dashboard (Vital Signs)")
    if show:
        try:
            fig.show()
        except Exception as exc:  # pragma: no cover - no display in CI
            logger.info(
                "dldiagnostics.checkpoint.show_skipped",
                extra={"error": str(exc)},
            )

    findings = diag.report()
    return diag, findings


def diagnose_classifier(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Classifier",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` cross-entropy classifiers.

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.cross_entropy(model(x), y)``. Use this for
    CNN, transformer, and transfer-learning exercises.

    Args:
        model: Trained classifier ``nn.Module``.
        dataloader: Yields ``(x, y)`` tuples; ``y`` is class indices.
        title: Title for the dashboard.
        n_batches: Batches to instrument.
        train_losses, val_losses: Optional epoch histories to replay.
        show: Show the figure inline.
        forward_returns_tuple: If ``True``, ``model(x)`` returns
            ``(logits, ...)`` (e.g. attention models) — only the first
            element is used as logits.

    Returns:
        ``(diag, findings)``
    """
    import torch.nn.functional as F  # local — torch already imported

    def _cls_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        logits = out[0] if forward_returns_tuple else out
        return F.cross_entropy(logits, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _cls_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


def diagnose_regressor(
    model: nn.Module,
    dataloader: DataLoader,
    *,
    title: str = "Regressor",
    n_batches: int = 8,
    train_losses: list[float] | None = None,
    val_losses: list[float] | None = None,
    show: bool = True,
    forward_returns_tuple: bool = False,
) -> tuple[DLDiagnostics, dict[str, Any]]:
    """Convenience wrapper for ``(x, y)`` MSE regressors (RNN exercises).

    Equivalent to :func:`run_diagnostic_checkpoint` with a built-in
    ``loss_fn`` that calls ``F.mse_loss(model(x), y)``.

    Args:
        forward_returns_tuple: Set ``True`` for attention models that
            return ``(predictions, attn_weights)``.
    """
    import torch.nn.functional as F

    def _reg_loss(m: nn.Module, batch: Any) -> torch.Tensor:
        x, y = batch[0], batch[1]
        out = m(x)
        pred = out[0] if forward_returns_tuple else out
        return F.mse_loss(pred, y)

    return run_diagnostic_checkpoint(
        model,
        dataloader,
        _reg_loss,
        title=title,
        n_batches=n_batches,
        train_losses=train_losses,
        val_losses=val_losses,
        show=show,
    )


__all__ = [
    "DLDiagnostics",
    "run_diagnostic_checkpoint",
    "diagnose_classifier",
    "diagnose_regressor",
]


# ── shared/mlfp05/ex_5.py ──
"""
Shared utilities for Exercise 5 — GANs and Generative Models.

Infrastructure only: data loading, visualisation helpers, metric computation,
and kailash-ml engine setup. No domain logic or business scenarios.
"""

import asyncio
import pickle
from pathlib import Path
from typing import TYPE_CHECKING

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import matplotlib.pyplot as plt

from kailash.db import ConnectionManager
from kailash_ml import ModelVisualizer
from kailash_ml.engines.experiment_tracker import ExperimentTracker
from kailash_ml.engines.model_registry import ModelRegistry
from kailash_ml.types import MetricSpec


if TYPE_CHECKING:
    from kailash_ml.engines.model_registry import ModelVersion

# ════════════════════════════════════════════════════════════════════════
# Constants
# ════════════════════════════════════════════════════════════════════════
LATENT_DIM = 64
IMG_DIM = 28 * 28
BATCH_SIZE = 128
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "mnist"
OUTPUT_DIR = Path(__file__).resolve().parent


# ════════════════════════════════════════════════════════════════════════
# Environment and device
# ════════════════════════════════════════════════════════════════════════
def init_environment() -> torch.device:
    """Set up environment, seeds, and return the compute device."""
    setup_environment()
    torch.manual_seed(42)
    np.random.seed(42)
    device = get_device()
    print(f"Using device: {device}")
    return device


# ════════════════════════════════════════════════════════════════════════
# Data loading
# ════════════════════════════════════════════════════════════════════════
def load_mnist(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, DataLoader]:
    """Load full MNIST (60K), scale to [-1, 1] for tanh generators.

    Returns:
        X_real: (60000, 1, 28, 28) tensor on device, range [-1, 1]
        y_real: (60000,) long tensor on device
        real_loader: DataLoader with batch_size=128, shuffle, drop_last
    """
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    train_set = torchvision.datasets.MNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_real = torch.stack([train_set[i][0] for i in range(len(train_set))])
    y_real = torch.tensor(
        [train_set[i][1] for i in range(len(train_set))], dtype=torch.long
    )
    X_real = (X_real * 2.0 - 1.0).to(device)
    y_real = y_real.to(device)

    print(
        f"MNIST: {len(X_real)} digits, shape {tuple(X_real.shape[1:])}, "
        f"pixel range [{X_real.min():.2f}, {X_real.max():.2f}]"
    )
    class_dist = ", ".join(f"{c}={int((y_real == c).sum())}" for c in range(10))
    print(f"  class distribution: {class_dist}")

    real_loader = DataLoader(
        TensorDataset(X_real), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
    )
    return X_real, y_real, real_loader


# ════════════════════════════════════════════════════════════════════════
# Kailash engine setup
# ════════════════════════════════════════════════════════════════════════
def setup_engines() -> (
    tuple[ConnectionManager, ExperimentTracker, str, ModelRegistry | None]
):
    """Create ExperimentTracker and ModelRegistry for GAN experiments.

    Returns:
        conn, tracker, experiment_name, registry (or None)
    """

    async def _setup():
        conn = ConnectionManager("sqlite:///mlfp05_gans.db")
        await conn.initialize()
        tracker = ExperimentTracker(conn)
        exp_name = await tracker.create_experiment(
            name="m5_gans",
            description="GAN variants on MNIST (60K images)",
        )
        try:
            registry = ModelRegistry(conn)
        except Exception as e:
            registry = None
            print(f"  Note: ModelRegistry setup skipped ({e})")
        return conn, tracker, exp_name, registry

    return asyncio.run(_setup())


async def close_engines(conn: ConnectionManager) -> None:
    """Cleanly shut down the connection manager."""
    await conn.close()


# ════════════════════════════════════════════════════════════════════════
# Generator and Discriminator architectures
# ════════════════════════════════════════════════════════════════════════
class Generator(nn.Module):
    """MLP Generator: z -> 784-d -> (1, 28, 28).

    Uses BatchNorm + LeakyReLU (DCGAN best practices) and Tanh output
    to match the [-1, 1] image scaling.
    """

    def __init__(self, latent_dim: int = LATENT_DIM, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden * 2),
            nn.BatchNorm1d(hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden * 2, IMG_DIM),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    """MLP Discriminator: 28x28 -> scalar logit.

    Dropout prevents D from overfitting to real images (memorising
    instead of learning distributional features).
    """

    def __init__(self, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(IMG_DIM, hidden * 2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ════════════════════════════════════════════════════════════════════════
# FID feature extractor
# ════════════════════════════════════════════════════════════════════════
class LeNetFeatureExtractor(nn.Module):
    """CNN feature extractor for FID computation.

    Returns 64-dim feature vectors (analogous to InceptionV3 pool3 layer).
    We use a domain-specific extractor because InceptionV3 expects 299x299 RGB;
    for 28x28 grayscale MNIST a trained LeNet gives more meaningful distances.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=2, padding=2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(64, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


def train_feature_extractor(
    X_real: torch.Tensor,
    y_real: torch.Tensor,
    device: torch.device,
    epochs: int = 5,
) -> LeNetFeatureExtractor:
    """Train the LeNet feature extractor on MNIST for FID computation.

    Returns the trained extractor in eval mode.
    """
    print("\n== Training feature extractor (for FID + mode coverage) ==")
    extractor = LeNetFeatureExtractor().to(device)
    opt = torch.optim.Adam(extractor.parameters(), lr=1e-3)
    X_01 = (X_real + 1.0) / 2.0  # [0, 1] for classifier

    for epoch in range(epochs):
        losses = []
        for xb, yb in DataLoader(
            TensorDataset(X_01, y_real), batch_size=256, shuffle=True
        ):
            loss = F.cross_entropy(extractor(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        with torch.no_grad():
            acc = (extractor(X_01[:10000]).argmax(-1) == y_real[:10000]).float().mean()
        print(f"  epoch {epoch+1}/{epochs}  loss={np.mean(losses):.3f}  acc={acc:.3f}")

    extractor.eval()
    return extractor


# ════════════════════════════════════════════════════════════════════════
# FID computation
# ════════════════════════════════════════════════════════════════════════
def compute_fid(
    extractor: LeNetFeatureExtractor,
    real: torch.Tensor,
    generated: torch.Tensor,
) -> float:
    """Frechet Inception Distance between real and generated images.

    FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r @ Sigma_g))

    Lower FID = closer to real distribution = better generator.
    Uses eigendecomposition (no scipy dependency).
    """
    extractor.eval()

    def _extract(images: torch.Tensor) -> np.ndarray:
        feats = []
        with torch.no_grad():
            for i in range(0, len(images), 512):
                feats.append(
                    extractor.extract_features(images[i : i + 512]).cpu().numpy()
                )
        return np.concatenate(feats)

    rf, gf = _extract(real), _extract(generated)
    mu_r, mu_g = rf.mean(0), gf.mean(0)
    sig_r, sig_g = np.cov(rf, rowvar=False), np.cov(gf, rowvar=False)

    diff = mu_r - mu_g
    product = sig_r @ sig_g
    eigvals, eigvecs = np.linalg.eigh(product)
    eigvals = np.maximum(eigvals, 0.0)
    sqrt_prod = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T

    return float(diff @ diff + np.trace(sig_r + sig_g - 2 * sqrt_prod))


# ════════════════════════════════════════════════════════════════════════
# Mode coverage diagnostic
# ════════════════════════════════════════════════════════════════════════
def mode_coverage(
    G: Generator,
    classifier: LeNetFeatureExtractor,
    device: torch.device,
    n: int = 5000,
) -> tuple[int, dict[int, int], float]:
    """Measure mode coverage: how many of the 10 digit classes the generator produces.

    Returns:
        n_classes: number of unique classes generated
        per_class_counts: dict mapping class -> count
        shannon_entropy: diversity measure (max = log2(10) = 3.32 for uniform)
    """
    G.eval()
    classifier.eval()
    with torch.no_grad():
        fake_01 = (G(torch.randn(n, LATENT_DIM, device=device)) + 1) / 2
        preds = classifier(fake_01).argmax(-1).cpu().numpy()
    unique, counts = np.unique(preds, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs + 1e-10)))
    return int(len(unique)), {int(k): int(v) for k, v in zip(unique, counts)}, entropy


# ════════════════════════════════════════════════════════════════════════
# Visualisation helpers
# ════════════════════════════════════════════════════════════════════════
def plot_image_grid(
    images: torch.Tensor,
    nrow: int = 8,
    ncol: int = 8,
    title: str = "Generated Images",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot an 8x8 grid of generated images.

    Args:
        images: (N, 1, 28, 28) tensor in [-1, 1] range
        nrow, ncol: grid dimensions
        title: figure title
        save_path: optional path to save the figure
    """
    n = min(nrow * ncol, len(images))
    fig, axes = plt.subplots(nrow, ncol, figsize=(12, 12))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    for i in range(nrow * ncol):
        ax = axes[i // ncol][i % ncol]
        if i < n:
            img = images[i].detach().cpu().squeeze().numpy()
            img = (img + 1) / 2  # [-1,1] -> [0,1]
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_latent_interpolation(
    G: Generator,
    device: torch.device,
    n_steps: int = 10,
    n_rows: int = 5,
    title: str = "Latent Space Interpolation",
    save_path: str | None = None,
) -> plt.Figure:
    """Interpolate between pairs of random latent vectors.

    Shows smooth transitions between generated images — evidence that
    the generator has learned a continuous manifold, not memorised digits.
    """
    G.eval()
    fig, axes = plt.subplots(n_rows, n_steps, figsize=(n_steps * 1.2, n_rows * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    with torch.no_grad():
        for row in range(n_rows):
            z1 = torch.randn(1, LATENT_DIM, device=device)
            z2 = torch.randn(1, LATENT_DIM, device=device)
            for col in range(n_steps):
                alpha = col / (n_steps - 1)
                z = (1 - alpha) * z1 + alpha * z2
                img = G(z).squeeze().cpu().numpy()
                img = (img + 1) / 2
                axes[row][col].imshow(img, cmap="gray", vmin=0, vmax=1)
                axes[row][col].axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_training_progression(
    G: Generator,
    device: torch.device,
    epoch_snapshots: dict[int, dict],
    title: str = "Training Progression",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot generated images at different training epochs.

    Shows how generation quality improves over training — from random
    noise to recognisable digits.

    Args:
        epoch_snapshots: dict of {epoch: state_dict} captured during training
    """
    n_snapshots = len(epoch_snapshots)
    n_samples = 8
    fig, axes = plt.subplots(
        n_snapshots, n_samples, figsize=(n_samples * 1.4, n_snapshots * 1.6)
    )
    fig.suptitle(title, fontsize=14, fontweight="bold")

    fixed_z = torch.randn(n_samples, LATENT_DIM, device=device)

    for row, (epoch, state_dict) in enumerate(sorted(epoch_snapshots.items())):
        G.load_state_dict(state_dict)
        G.eval()
        with torch.no_grad():
            imgs = G(fixed_z)
        for col in range(n_samples):
            ax = axes[row][col] if n_snapshots > 1 else axes[col]
            img = imgs[col].squeeze().cpu().numpy()
            img = (img + 1) / 2
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"Epoch {epoch}", fontsize=10, rotation=0, labelpad=50)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


def plot_loss_curves(
    g_losses: list[float],
    d_losses: list[float],
    title: str = "Training Dynamics",
    g_label: str = "Generator",
    d_label: str = "Discriminator",
    save_path: str | None = None,
) -> plt.Figure:
    """Plot G vs D loss curves across epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    epochs = range(1, len(g_losses) + 1)

    # Individual losses
    ax1.plot(epochs, g_losses, "b-", linewidth=2, label=g_label)
    ax1.plot(epochs, d_losses, "r-", linewidth=2, label=d_label)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("G vs D Loss", fontsize=13)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    # G/D ratio — healthy GAN has ratio near 1
    ratio = [g / (d + 1e-8) for g, d in zip(g_losses, d_losses)]
    ax2.plot(epochs, ratio, "g-", linewidth=2)
    ax2.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Balanced (1.0)")
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("G/D Loss Ratio", fontsize=12)
    ax2.set_title("Training Balance", fontsize=13)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    return fig


# ════════════════════════════════════════════════════════════════════════
# Model registration helper
# ════════════════════════════════════════════════════════════════════════
def register_generator(
    registry: ModelRegistry | None,
    name: str,
    model: Generator,
    fid: float,
    coverage: int,
    entropy: float,
) -> "ModelVersion | None":
    """Register a trained generator in ModelRegistry with quality metrics."""
    if registry is None:
        print(f"  ModelRegistry not available — skipping {name}")
        return None

    async def _register():
        ver = await registry.register_model(
            name=f"m5_{name}",
            artifact=pickle.dumps(model.state_dict()),
            metrics=[
                MetricSpec(name="fid_score", value=fid),
                MetricSpec(name="mode_coverage", value=float(coverage)),
                MetricSpec(name="class_entropy", value=entropy),
            ],
        )
        print(
            f"  Registered {name}: v={ver.version}, FID={fid:.2f}, "
            f"coverage={coverage}/10"
        )
        return ver

    return asyncio.run(_register())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 5.3: GAN Evaluation and Model Registry
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why "looks good" is not a valid evaluation metric for GANs
#   - FID (Frechet Inception Distance) — the standard automated metric
#   - Mode coverage analysis — detecting hidden mode collapse
#   - Shannon entropy as a diversity measure
#   - Register trained generators in ModelRegistry with quality metrics
#   - Build a quality assurance pipeline for synthetic data production
#   - Apply: QA validation for the insurance company's synthetic data
#     pipeline before deploying to production fraud detection models
#
# PREREQUISITES: ex_5/01_vanilla_gan.py, ex_5/02_wgan_gp.py
# ESTIMATED TIME: ~40 min
# DATASET: MNIST — 60,000 real 28x28 grayscale digits
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import os

import numpy as np
import torch
import matplotlib.pyplot as plt

from kailash_ml import ModelVisualizer



## PHASE 1 — THEORY: Why GAN Evaluation Is Hard


THE PROBLEM:
  Unlike classifiers (accuracy, F1) or regressors (MSE, R²), GANs have
  no single ground truth to compare against. A generator that produces
  perfect images of the digit "7" — and ONLY "7" — could score well on
  per-image quality but is useless for any practical application.

  "LOOKS GOOD" IS DANGEROUS:
  Human visual inspection doesn't scale, is subjective, and misses
  subtle distribution mismatches. A GAN producing sharp but repetitive
  images fools the eye while silently failing on diversity.

  THREE EVALUATION DIMENSIONS:

  1. QUALITY (per-image fidelity):
     Are individual generated images sharp and realistic?
     Metric: FID (lower = better)

  2. DIVERSITY (mode coverage):
     Does the generator cover the full range of real data?
     Metric: mode coverage count + Shannon entropy

  3. NOVELTY (not memorising):
     Is the generator creating NEW images, not copying training data?
     Metric: nearest-neighbour distance to training set

  FID — FRECHET INCEPTION DISTANCE:

  FID treats both real and generated images as points in a learned
  feature space (from a pre-trained classifier). It then fits a
  multivariate Gaussian to each set and measures the distance between
  the two Gaussians:

    FID = ||mu_r - mu_g||^2 + Tr(Sig_r + Sig_g - 2*sqrt(Sig_r @ Sig_g))

  Intuition for professionals:
  - FID = 0: generated images are statistically indistinguishable from real
  - FID < 10: publication-quality generation
  - FID 10-50: recognisable but imperfect
  - FID 50-100: blurry or distorted
  - FID > 100: the generator hasn't learned much

  MODE COLLAPSE DETECTION:

  Mode collapse is the GAN equivalent of a factory that produces only
  one product. Shannon entropy quantifies diversity:
  - max entropy = log2(10) = 3.32 (uniform across all 10 digit classes)
  - entropy = 0: generator produces only one class (total collapse)

  A generator can have low FID (individual images look good) but low
  entropy (it only produces 3 of the 10 digit types). Both metrics
  are needed.


In [ ]:
print("=" * 70)
print("  PHASE 1 — THEORY: The GAN Evaluation Problem")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: FID Computation Pipeline + Feature Extractor


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 2 — BUILD: FID Pipeline and Feature Extractor")
print("=" * 70)

device = init_environment()
X_real, y_real, real_loader = load_mnist(device)
conn, tracker, exp_name, registry = setup_engines()

# Train the feature extractor for FID computation
fid_ext = train_feature_extractor(X_real, y_real, device, epochs=5)



### Checkpoint 1


In [ ]:
fid_ext.eval()
with torch.no_grad():
    _test_acc = (
        (fid_ext((X_real[:1000] + 1) / 2).argmax(-1) == y_real[:1000]).float().mean()
    )
assert _test_acc > 0.8, f"Feature extractor accuracy too low: {_test_acc:.3f}"
print(f"  Feature extractor test accuracy: {_test_acc:.1%}")
print("\n--- Checkpoint 1 passed --- feature extractor trained\n")



## PHASE 3 — TRAIN: Both GAN Variants for Comparative Evaluation


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 3 — TRAIN: Vanilla GAN + WGAN-GP for Comparison")
print("=" * 70)

import torch.nn as nn


# TODO: Train vanilla GAN (15 epochs) with BCE loss
# Hint: Same pattern as ex_5/01 — Generator, Discriminator, Adam(0.5, 0.999),
#       BCEWithLogitsLoss, alternate D and G training each batch
print("\n  Training Vanilla GAN (15 epochs)...")
G_gan = Generator().to(device)
D_gan = Discriminator().to(device)
opt_g_gan = torch.optim.Adam(G_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_d_gan = torch.optim.Adam(D_gan.parameters(), lr=2e-4, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()
gan_g_losses, gan_d_losses = [], []

for epoch in range(15):
    eg, ed = [], []
    for (real_batch,) in real_loader:
        bs = real_batch.size(0)
        z = torch.randn(bs, LATENT_DIM, device=device)
        fake = G_gan(z).detach()
        # TODO: D loss = BCE on real (target=1) + BCE on fake (target=0)
        # Hint: loss_d = bce(D_gan(real_batch), torch.ones(bs, 1, device=device))
        #              + bce(D_gan(fake), torch.zeros(bs, 1, device=device))
        loss_d = ____
        opt_d_gan.zero_grad()
        loss_d.backward()
        opt_d_gan.step()
        z = torch.randn(bs, LATENT_DIM, device=device)
        # TODO: G loss = fool D by labelling fakes as real
        # Hint: loss_g = bce(D_gan(G_gan(z)), torch.ones(bs, 1, device=device))
        loss_g = ____
        opt_g_gan.zero_grad()
        loss_g.backward()
        opt_g_gan.step()
        eg.append(loss_g.item())
        ed.append(loss_d.item())
    gan_g_losses.append(float(np.mean(eg)))
    gan_d_losses.append(float(np.mean(ed)))
    print(
        f"  [Vanilla] epoch {epoch+1:2d}/15  "
        f"D={gan_d_losses[-1]:.3f}  G={gan_g_losses[-1]:.3f}"
    )


# TODO: Implement gradient penalty function for WGAN-GP
# Hint: Same as ex_5/02 — interpolate real+fake, compute grad norm, penalise != 1
def gradient_penalty(D, real, fake):
    batch = real.size(0)
    alpha = torch.rand(batch, 1, 1, 1, device=real.device)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp = D(interp)
    grad = torch.autograd.grad(
        outputs=d_interp,
        inputs=interp,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    # TODO: Return gradient penalty = mean of (||grad||_2 - 1)^2
    # Hint: ((grad.reshape(batch, -1).norm(2, dim=1) - 1) ** 2).mean()
    return ____


# TODO: Train WGAN-GP (20 epochs) with critic training
# Hint: 5 critic steps per G step, Adam(0.5, 0.9), lr=1e-4
#       Critic loss = D(fake).mean() - D(real).mean() + 10.0 * gp
#       G loss = -D(G(z)).mean()
print("\n  Training WGAN-GP (20 epochs)...")
G_wgan = Generator().to(device)
D_wgan = Discriminator().to(device)
opt_g_wgan = torch.optim.Adam(G_wgan.parameters(), lr=1e-4, betas=(0.5, 0.9))
opt_d_wgan = torch.optim.Adam(D_wgan.parameters(), lr=1e-4, betas=(0.5, 0.9))
wgan_g_losses, wgan_d_losses = [], []

for epoch in range(20):
    eg, ed = [], []
    for (real_batch,) in real_loader:
        bs = real_batch.size(0)
        for _ in range(5):
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake = G_wgan(z).detach()
            gp = gradient_penalty(D_wgan, real_batch, fake)
            # TODO: Wasserstein critic loss + gradient penalty
            # Hint: loss_d = D_wgan(fake).mean() - D_wgan(real_batch).mean() + 10.0 * gp
            loss_d = ____
            opt_d_wgan.zero_grad()
            loss_d.backward()
            opt_d_wgan.step()
        z = torch.randn(bs, LATENT_DIM, device=device)
        # TODO: G loss — maximise critic score on fakes
        # Hint: loss_g = -D_wgan(G_wgan(z)).mean()
        loss_g = ____
        opt_g_wgan.zero_grad()
        loss_g.backward()
        opt_g_wgan.step()
        eg.append(loss_g.item())
        ed.append(loss_d.item())
    wgan_g_losses.append(float(np.mean(eg)))
    wgan_d_losses.append(float(np.mean(ed)))
    print(
        f"  [WGAN-GP] epoch {epoch+1:2d}/20  "
        f"critic={wgan_d_losses[-1]:.3f}  G={wgan_g_losses[-1]:.3f}"
    )



### Checkpoint 2


In [ ]:
assert len(gan_g_losses) == 15, "Vanilla GAN should train 15 epochs"
assert len(wgan_g_losses) == 20, "WGAN-GP should train 20 epochs"
print("\n--- Checkpoint 2 passed --- both generators trained\n")



## Compute FID scores


In [ ]:
print("\n  Computing FID scores...")

N_FID = 10000
G_gan.eval()
G_wgan.eval()

with torch.no_grad():
    # TODO: Generate fake images from both generators (N_FID each), scale to [0, 1]
    # Hint: gan_fake_01 = (G_gan(torch.randn(N_FID, LATENT_DIM, device=device)) + 1) / 2
    #       wgan_fake_01 = (G_wgan(torch.randn(N_FID, LATENT_DIM, device=device)) + 1) / 2
    gan_fake_01 = ____
    wgan_fake_01 = ____

rng = np.random.default_rng(42)
real_sub = (X_real[rng.choice(len(X_real), N_FID, replace=False)] + 1) / 2

# TODO: Compute FID for both generators using the trained feature extractor
# Hint: fid_gan = compute_fid(fid_ext, real_sub, gan_fake_01)
#       fid_wgan = compute_fid(fid_ext, real_sub, wgan_fake_01)
fid_gan = ____
fid_wgan = ____

print(f"\n  FID Scores (lower = better):")
print(f"    Vanilla GAN: {fid_gan:.2f}")
print(f"    WGAN-GP:     {fid_wgan:.2f}")

# TODO: Compute mode coverage for both generators
# Hint: cov_gan, dist_gan, ent_gan = mode_coverage(G_gan, fid_ext, device)
#       cov_wgan, dist_wgan, ent_wgan = mode_coverage(G_wgan, fid_ext, device)
cov_gan, dist_gan, ent_gan = ____
cov_wgan, dist_wgan, ent_wgan = ____

print(f"\n  Mode Coverage (all 10 digit classes):")
print(f"    Vanilla GAN: {cov_gan}/10 classes, entropy={ent_gan:.2f}/3.32")
print(f"    Distribution: {dist_gan}")
print(f"    WGAN-GP:     {cov_wgan}/10 classes, entropy={ent_wgan:.2f}/3.32")
print(f"    Distribution: {dist_wgan}")


# Log to ExperimentTracker
async def _log_evaluation():
    async with tracker.run(experiment_name=exp_name, run_name="fid_evaluation") as ctx:
        await ctx.log_param("n_generated", str(N_FID))
        await ctx.log_metrics(
            {
                "fid_vanilla_gan": fid_gan,
                "fid_wgan_gp": fid_wgan,
                "coverage_vanilla": float(cov_gan),
                "coverage_wgan": float(cov_wgan),
                "entropy_vanilla": ent_gan,
                "entropy_wgan": ent_wgan,
            }
        )


await _log_evaluation()



### Checkpoint 3


In [ ]:
assert fid_gan >= 0 and fid_wgan >= 0, "FID must be non-negative"
assert 0 <= ent_gan <= np.log2(10) + 0.01, "Entropy out of range"
assert cov_gan >= 1 and cov_wgan >= 1, "Must produce at least 1 class"
# INTERPRETATION: FID = 0 means identical distributions. Typical MNIST
# GAN FID after a few epochs: 10-100. Production papers target FID < 10.
# WGAN-GP typically achieves better coverage because Wasserstein distance
# provides gradients even when distributions don't overlap.
print("\n--- Checkpoint 3 passed --- FID and mode coverage computed\n")



## PHASE 4 — VISUALISE: FID Comparison, Mode Coverage, Latent Walk


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 4 — VISUALISE: Evaluation Metrics Dashboard")
print("=" * 70)

# 4A: FID score comparison bar chart
print("\n  4A: FID score comparison")
# TODO: Create bar chart comparing FID scores of both generators
# Hint: Use plt.subplots, ax.bar with ["Vanilla GAN", "WGAN-GP"], [fid_gan, fid_wgan]
#       Add reference lines at FID=10 (publication), 50 (recognisable), 100 (poor)
fig_fid, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ["Vanilla GAN", "WGAN-GP"],
    [fid_gan, fid_wgan],
    color=["#e74c3c", "#2ecc71"],
    width=0.5,
    edgecolor="black",
    linewidth=1.2,
)
ax.set_ylabel("FID Score (lower = better)", fontsize=13)
ax.set_title(
    "Frechet Inception Distance Comparison\n"
    f"(computed on {N_FID:,} generated vs {N_FID:,} real images)",
    fontsize=14,
    fontweight="bold",
)
# TODO: Add value labels on bars and reference lines
# Hint: ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.5, f"{val:.1f}", ...)
#       ax.axhline(y=10, color="green", linestyle="--", alpha=0.5, label="Publication quality")
for bar, val in zip(bars, [fid_gan, fid_wgan]):
    ____  # TODO: Add value label text on each bar
ax.axhline(
    y=10, color="green", linestyle="--", alpha=0.5, label="Publication quality (<10)"
)
ax.axhline(
    y=50, color="orange", linestyle="--", alpha=0.5, label="Recognisable (10-50)"
)
ax.axhline(y=100, color="red", linestyle="--", alpha=0.5, label="Poor quality (>100)")
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
fig_fid.savefig(
    str(OUTPUT_DIR / "ex_5_03_fid_comparison.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# 4B: Mode coverage matrix — heatmap of generated digit classes
print("\n  4B: Mode coverage matrix (all 10 digit classes)")
# TODO: Create side-by-side bar charts showing per-digit class distribution
# Hint: For each GAN, compute counts per class 0-9, convert to percentages,
#       bar chart with green (present) / red (missing) colours
fig_mode, axes = plt.subplots(1, 2, figsize=(16, 5))
fig_mode.suptitle(
    "Mode Coverage: Which Digits Does Each GAN Generate?",
    fontsize=14,
    fontweight="bold",
)

for idx, (name, dist, ent, cov) in enumerate(
    [
        ("Vanilla GAN", dist_gan, ent_gan, cov_gan),
        ("WGAN-GP", dist_wgan, ent_wgan, cov_wgan),
    ]
):
    counts = [dist.get(c, 0) for c in range(10)]
    total = sum(counts)
    pcts = [c / total * 100 if total > 0 else 0 for c in counts]
    colors = ["#2ecc71" if c > 0 else "#e74c3c" for c in counts]

    bars = axes[idx].bar(
        range(10), pcts, color=colors, edgecolor="black", linewidth=0.8
    )
    axes[idx].set_xlabel("Digit Class", fontsize=12)
    axes[idx].set_ylabel("% of Generated Images", fontsize=12)
    axes[idx].set_title(
        f"{name}\n{cov}/10 classes, entropy={ent:.2f}/3.32", fontsize=13
    )
    axes[idx].set_xticks(range(10))
    axes[idx].axhline(
        y=10, color="blue", linestyle="--", alpha=0.5, label="Ideal uniform (10%)"
    )
    axes[idx].legend(fontsize=10)
    axes[idx].grid(True, alpha=0.2, axis="y")

    # TODO: Add percentage labels on each bar
    # Hint: for bar, pct in zip(bars, pcts):
    #           if pct > 0: axes[idx].text(bar.get_x()+bar.get_width()/2, ...)
    ____

plt.tight_layout()
fig_mode.savefig(
    str(OUTPUT_DIR / "ex_5_03_mode_coverage.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# 4C: Latent space interpolation walk — smooth transitions
print("\n  4C: Latent space interpolation (WGAN-GP)")
fig_walk = plot_latent_interpolation(
    G_wgan,
    device,
    n_steps=12,
    n_rows=6,
    title="WGAN-GP Latent Walk — Smooth Digit Transitions",
    save_path=str(OUTPUT_DIR / "ex_5_03_latent_walk.png"),
)
plt.show()

# 4D: Combined training dynamics
print("\n  4D: Combined training dynamics comparison")
viz = ModelVisualizer()
fig_all = viz.training_history(
    metrics={
        "Vanilla G": gan_g_losses,
        "Vanilla D": gan_d_losses,
        "WGAN G": wgan_g_losses,
        "WGAN Critic": wgan_d_losses,
    },
    x_label="Epoch",
    y_label="Loss",
)
fig_all.write_html(str(OUTPUT_DIR / "ex_5_03_combined_training.html"))
print("  Interactive combined training curves saved")

# 4E: Comprehensive evaluation dashboard
print("\n  4E: Evaluation dashboard")
# TODO: Create a 2x2 dashboard with FID comparison, entropy comparison,
#       G loss curves, and D/Critic loss curves
# Hint: fig_dash, axes = plt.subplots(2, 2, figsize=(16, 12))
fig_dash, axes = plt.subplots(2, 2, figsize=(16, 12))
fig_dash.suptitle(
    "GAN Evaluation Dashboard — Vanilla GAN vs WGAN-GP",
    fontsize=16,
    fontweight="bold",
)

# TODO: Top-left — FID bar chart
# Hint: axes[0][0].bar(["Vanilla GAN", "WGAN-GP"], [fid_gan, fid_wgan], ...)
____

# TODO: Top-right — Entropy bar chart with max entropy line
# Hint: axes[0][1].bar(["Vanilla GAN", "WGAN-GP"], [ent_gan, ent_wgan], ...)
#       axes[0][1].axhline(y=np.log2(10), ...)
____

# TODO: Bottom-left — G loss curves for both generators
# Hint: axes[1][0].plot(range(1, 16), gan_g_losses, "r-", ...)
#       axes[1][0].plot(range(1, 21), wgan_g_losses, "g-", ...)
____

# TODO: Bottom-right — D/Critic loss curves for both generators
# Hint: axes[1][1].plot(range(1, 16), gan_d_losses, "r-", ...)
#       axes[1][1].plot(range(1, 21), wgan_d_losses, "g-", ...)
____

plt.tight_layout()
fig_dash.savefig(
    str(OUTPUT_DIR / "ex_5_03_evaluation_dashboard.png"), dpi=150, bbox_inches="tight"
)
plt.show()



### Checkpoint 4


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_fid_comparison.png")
), "FID comparison should exist"
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_evaluation_dashboard.png")
), "Dashboard should exist"
print("\n--- Checkpoint 4 passed --- evaluation visualisations complete\n")



## Register generators in ModelRegistry


In [ ]:
print("\n  Registering generators in ModelRegistry...")
ver_gan = register_generator(
    registry, "dcgan_generator", G_gan, fid_gan, cov_gan, ent_gan
)
ver_wgan = register_generator(
    registry, "wgan_gp_generator", G_wgan, fid_wgan, cov_wgan, ent_wgan
)



### Checkpoint 5


In [ ]:
if registry is not None:
    assert ver_gan is not None, "Vanilla GAN should be registered"
    assert ver_wgan is not None, "WGAN-GP should be registered"
print("\n--- Checkpoint 5 passed --- generators registered\n")



## PHASE 5 — APPLY: QA Pipeline for Insurance Synthetic Data


BUSINESS SCENARIO:
  You are the ML engineering lead at the Singapore insurance company
  from Exercise 5.1. Your team has trained a WGAN-GP to generate
  synthetic policyholder profiles for fraud detection model training.

  Before deploying synthetic data to production, the Chief Risk Officer
  (CRO) asks: "How do you KNOW the synthetic data is good enough?
  What if the generator produces biased profiles that make the fraud
  model miss certain claim types?"

  YOUR QA PIPELINE:
  1. FID threshold: synthetic data must score below FID 50
     (statistically close to real distribution)
  2. Mode coverage: all major claim categories must be represented
     (no category can be below 5% of total)
  3. Distribution matching: key statistical properties must match
     (mean, variance, correlation structure)
  4. Downstream model validation: fraud model trained on synthetic
     data must achieve within 5% of real-data baseline accuracy

  This is the production gate — synthetic data that fails any check
  is rejected and the generator is retrained or tuned.


In [ ]:
print("\n" + "=" * 70)
print("  PHASE 5 — APPLY: Synthetic Data QA for Insurance Production")
print("=" * 70)
print(
)

# Step 1: Define QA thresholds
FID_THRESHOLD = 50.0
MIN_MODE_COVERAGE = 8  # At least 8/10 classes
MIN_ENTROPY = 2.5  # Minimum diversity (max is 3.32)
MIN_CLASS_PCT = 3.0  # No class below 3% of total

print("  QA Thresholds:")
print(f"    FID score:        < {FID_THRESHOLD}")
print(f"    Mode coverage:    >= {MIN_MODE_COVERAGE}/10 classes")
print(f"    Shannon entropy:  >= {MIN_ENTROPY}/3.32")
print(f"    Min class share:  >= {MIN_CLASS_PCT}%")

# Step 2: Run QA on both generators
print("\n  Running QA pipeline on both generators...")

# TODO: Evaluate both generators against QA thresholds
# Hint: For each generator, check fid < threshold, coverage >= min,
#       entropy >= min, and min class percentage >= threshold
qa_results = {}
for name, G, fid, cov, dist, ent in [
    ("Vanilla GAN", G_gan, fid_gan, cov_gan, dist_gan, ent_gan),
    ("WGAN-GP", G_wgan, fid_wgan, cov_wgan, dist_wgan, ent_wgan),
]:
    total = sum(dist.values())
    min_pct = min(dist.values()) / total * 100 if total > 0 and dist else 0.0

    # TODO: Check each QA criterion
    # Hint: fid_pass = fid < FID_THRESHOLD
    #       cov_pass = cov >= MIN_MODE_COVERAGE
    #       ent_pass = ent >= MIN_ENTROPY
    #       pct_pass = min_pct >= MIN_CLASS_PCT
    fid_pass = ____
    cov_pass = ____
    ent_pass = ____
    pct_pass = ____

    all_pass = fid_pass and cov_pass and ent_pass and pct_pass

    qa_results[name] = {
        "fid": fid,
        "fid_pass": fid_pass,
        "coverage": cov,
        "cov_pass": cov_pass,
        "entropy": ent,
        "ent_pass": ent_pass,
        "min_class_pct": min_pct,
        "pct_pass": pct_pass,
        "overall": all_pass,
    }

# Step 3: QA results visualisation
# TODO: Create normalised bar chart showing % of QA threshold met
# Hint: For FID (lower=better), normalise as threshold/value * 100
#       For others (higher=better), normalise as value/threshold * 100
fig_qa, ax = plt.subplots(figsize=(14, 7))
fig_qa.suptitle(
    "Synthetic Data QA Pipeline Results\n"
    "Production Gate for Insurance Fraud Model Training",
    fontsize=14,
    fontweight="bold",
)

categories = [
    "FID Score\n(< 50)",
    "Mode Coverage\n(>= 8/10)",
    "Entropy\n(>= 2.5)",
    "Min Class %\n(>= 3%)",
]
vanilla_scores = [
    qa_results["Vanilla GAN"]["fid"],
    qa_results["Vanilla GAN"]["coverage"],
    qa_results["Vanilla GAN"]["entropy"],
    qa_results["Vanilla GAN"]["min_class_pct"],
]
wgan_scores = [
    qa_results["WGAN-GP"]["fid"],
    qa_results["WGAN-GP"]["coverage"],
    qa_results["WGAN-GP"]["entropy"],
    qa_results["WGAN-GP"]["min_class_pct"],
]
thresholds = [FID_THRESHOLD, MIN_MODE_COVERAGE, MIN_ENTROPY, MIN_CLASS_PCT]

x = np.arange(len(categories))
width = 0.3

# TODO: Normalise scores to percentage of threshold met
# Hint: For FID (lower=better): min(threshold / (value + 1e-8) * 100, 150)
#       For others (higher=better): min(value / threshold * 100, 150)
vanilla_norm = []
wgan_norm = []
for v, w, t, cat in zip(vanilla_scores, wgan_scores, thresholds, categories):
    if "FID" in cat:
        ____  # TODO: Append normalised FID (lower=better)
        ____
    else:
        ____  # TODO: Append normalised coverage/entropy/pct (higher=better)
        ____

bars1 = ax.bar(
    x - width / 2, vanilla_norm, width, label="Vanilla GAN", color="#e74c3c", alpha=0.8
)
bars2 = ax.bar(
    x + width / 2, wgan_norm, width, label="WGAN-GP", color="#2ecc71", alpha=0.8
)
ax.axhline(
    y=100, color="black", linestyle="--", linewidth=2, label="QA Threshold (100%)"
)
ax.set_ylabel("% of QA Threshold Met", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=11, loc="upper right")
ax.grid(True, alpha=0.2, axis="y")
ax.set_ylim(0, 160)

# TODO: Add PASS/FAIL indicators on each bar
# Hint: For each bar pair, check the pass/fail from qa_results,
#       add green "PASS" or red "FAIL" text above the bar
____

plt.tight_layout()
fig_qa.savefig(
    str(OUTPUT_DIR / "ex_5_03_qa_pipeline.png"), dpi=150, bbox_inches="tight"
)
plt.show()

# Step 4: Stakeholder-ready QA report
van_overall = qa_results["Vanilla GAN"]["overall"]
wgan_overall = qa_results["WGAN-GP"]["overall"]

print("\n  ┌────────────────────────────────────────────────────────────────┐")
print("  │  SYNTHETIC DATA QA REPORT — Insurance Fraud Model Pipeline    │")
print("  ├────────────────────────────────────────────────────────────────┤")
print("  │                                                                │")
print(f"  │  {'Metric':<22} {'Vanilla GAN':>13} {'WGAN-GP':>13} {'Threshold':>12} │")
print(f"  │  {'─'*60}  │")
print(f"  │  {'FID Score':<22} {fid_gan:>13.1f} {fid_wgan:>13.1f} {'< 50':>12} │")
print(
    f"  │  {'Mode Coverage':<22} {cov_gan:>12}/10 {cov_wgan:>12}/10 {'>= 8/10':>12} │"
)
print(
    f"  │  {'Shannon Entropy':<22} {ent_gan:>13.2f} {ent_wgan:>13.2f} {'>= 2.50':>12} │"
)
van_min = qa_results["Vanilla GAN"]["min_class_pct"]
wgan_min = qa_results["WGAN-GP"]["min_class_pct"]
print(
    f"  │  {'Min Class Share':<22} {van_min:>12.1f}% {wgan_min:>12.1f}% {'>= 3.0%':>12} │"
)
print("  │                                                                │")
van_status = "APPROVED" if van_overall else "REJECTED"
wgan_status = "APPROVED" if wgan_overall else "REJECTED"
print(f"  │  {'OVERALL STATUS':<22} {van_status:>13} {wgan_status:>13}              │")
print("  │                                                                │")
better = "WGAN-GP" if fid_wgan < fid_gan else "Vanilla GAN"
print(f"  │  Recommended generator: {better:<38} │")
print(f"  │  Best FID score: {min(fid_gan, fid_wgan):<44.1f} │")
print("  │                                                                │")
print("  │  DECISION: CRO can approve WGAN-GP synthetic data for         │")
print("  │  production fraud model training. QA pipeline will run         │")
print("  │  automatically on every new generator version via ModelRegistry│")
print("  └────────────────────────────────────────────────────────────────┘")



### Checkpoint 6


In [ ]:
assert os.path.exists(
    str(OUTPUT_DIR / "ex_5_03_qa_pipeline.png")
), "QA pipeline chart should exist"
print("\n--- Checkpoint 6 passed --- QA pipeline complete\n")



## Final Summary


In [ ]:
print("\n" + "=" * 70)
print("  EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\n  Experiment: {exp_name}")
print(f"  Dataset: MNIST (60,000 images), latent_dim={LATENT_DIM}")
print(f"\n  {'Metric':<25} {'Vanilla GAN':>14} {'WGAN-GP':>14}")
print(f"  {'-'*53}")
print(f"  {'Epochs':<25} {'15':>14} {'20':>14}")
print(f"  {'Final G loss':<25} {gan_g_losses[-1]:>14.3f} {wgan_g_losses[-1]:>14.3f}")
print(
    f"  {'Final D/Critic loss':<25} {gan_d_losses[-1]:>14.3f} {wgan_d_losses[-1]:>14.3f}"
)
print(f"  {'FID score':<25} {fid_gan:>14.2f} {fid_wgan:>14.2f}")
print(f"  {'Mode coverage':<25} {cov_gan:>13}/10 {cov_wgan:>13}/10")
print(f"  {'Class entropy':<25} {ent_gan:>14.2f} {ent_wgan:>14.2f}")
print(f"\n  Best generator by FID: {better}")



## Cleanup


In [ ]:
await close_engines(conn)



## REFLECTION


GAN EVALUATION METRICS:
  [x] FID (Frechet Inception Distance): the gold standard for GAN quality.
      Measures distributional distance in a learned feature space.
  [x] Mode coverage: counting how many classes the generator produces.
      A sharp but repetitive GAN fails this test.
  [x] Shannon entropy: quantifying generation diversity.
      Max = log2(10) = 3.32 for MNIST (uniform across all digits).
  [x] Minimum class share: no category can be underrepresented
      (prevents hidden bias in synthetic datasets).

  MODEL REGISTRY:
  [x] Registered both generators with FID + coverage + entropy metrics
  [x] Version tracking enables A/B comparison between generator versions
  [x] Promotion criteria: lower FID + higher coverage = promote to serving

  QA PIPELINE FOR PRODUCTION:
  [x] Defined quantitative thresholds (FID < 50, coverage >= 8/10, etc.)
  [x] Automated evaluation — no "looks good to me" subjectivity
  [x] Stakeholder-ready report for CRO approval
  [x] Pipeline runs on every new generator version in ModelRegistry

  REAL-WORLD APPLICATION:
  [x] Insurance synthetic data QA: proving data quality before production
  [x] CRO-ready quality report with pass/fail per metric
  [x] Quantified business risk: what happens if synthetic data is biased

  KEY INSIGHTS:
  - FID alone is not enough: a generator can have low FID on the modes
    it covers while completely missing other modes
  - Mode coverage alone is not enough: a generator can cover all modes
    but produce blurry, low-quality images
  - You need BOTH quality (FID) AND diversity (coverage + entropy)
  - In production, these checks run automatically on every new model
    version — the CRO never needs to "eyeball" synthetic data again

  WHEN TO USE WHICH GAN:
  - Vanilla GAN: quick prototyping, simple datasets, no stability needs
  - WGAN-GP: production use, medical/financial data, stability required
  - Both need the same evaluation pipeline — the QA doesn't change,
    only the generator architecture does.

  GAN vs VAE vs Diffusion (from M5 Exercise 1):
  - GANs:      Sharp images, hard to train, fast sampling
  - VAEs:      Blurry but stable, continuous latent space, fast
  - Diffusion: Sharp + stable, best quality, SLOW sampling


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `shared/mlfp05/diagnostics.py` — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.


def _diag_loss(m, batch):
    # GAN evaluation — diagnostic on the generator model
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (GAN Evaluation (FID + Inception Score)) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        generator,
        noise_loader,
        _diag_loss,
        title="GAN Evaluation (FID + Inception Score)",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [!] Gradient flow (WARNING): G gradients RMS 3.2e-03, D gradients RMS 4.5e-02
#     (G/D imbalance — D dominating is the canonical GAN failure mode).
# [!] Dead neurons  (WARNING): 28% saturation in G's final tanh layer —
#     generator is producing outputs clustered at [-1, 1] extremes.
# [?] Loss trend    (MIXED): D loss → 0.1 (winning), G loss climbing.
#     Textbook sign of D overpowering G — generator can't keep up.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST] G/D gradient imbalance is the signature of GAN training
#     collapse. When D's RMS >> G's RMS, the discriminator has "won" —
#     G is receiving useless gradient signal. Slide 5.5 shows this as
#     the most common GAN failure.
#     >> Prescription: reduce D learning rate OR train G for N steps
#        per 1 D step OR apply WGAN-GP (ex_5/02) which sidesteps this
#        via gradient penalty instead of BCE.
#
#  [X-RAY] 28% tanh saturation means the generator is producing
#     extreme outputs — diversity is collapsing. Combined with the
#     loss trend, this is mode collapse territory.
#     >> Prescription: add minibatch discrimination, feature matching,
#        or switch to WGAN which has no saturation problem.
#
#  [STETHOSCOPE] Diverging G/D losses (D→0, G→∞) is the classic
#     "Nash equilibrium lost" pattern. WGAN-GP (next exercise) is
#     the direct fix.

